# 04 — Classification

**Mini Project 2.** Predict the crowding outcome from the area profile. One
`Pipeline` throughout, so every preprocessing step stays inside
cross-validation.

Two comparisons are reported:

- **Models** — logistic baseline, KNN, random forest
- **Features** — area profile alone, plus cluster label, plus administrative area

The second is the one that answers the project's question.

## The metric, decided before the first model

A false negative — a crowded household the model misses — is the costly error
here, because the response it would prompt is support rather than a penalty.
So:

- **Headline:** recall on the crowded class
- **Alongside it:** PR-AUC, not ROC-AUC, because the positive class is the minority
- **Reported for completeness:** accuracy — never as the decision metric

The base rate is printed next to every result, because accuracy that does not
beat the base rate is not a model.

## Setup

In [ ]:
import sys, json
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mp2

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
np.random.seed(42)

df = pd.read_csv(mp2.OUT / "analysis_frame.csv")
spec = json.loads((mp2.OUT / "feature_spec.json").read_text())

NUMERIC = spec["numeric"]
CATEGORICAL = spec["categorical"]
TARGET = spec["target"]
LEAKY = spec["leaky_or_id"]

mp2.positive_rate_report(df[TARGET], TARGET)
print(f"\n{len(df):,} rows, {len(NUMERIC)} numeric, {len(CATEGORICAL)} categorical")

In [ ]:
labels = pd.read_csv(mp2.OUT / "cluster_labels.csv")
df = df.merge(labels, on="sa2_code", how="left")
print("cluster label coverage:", df["cluster"].notna().mean().round(4))

## Train / test split

Stratified on the target, fixed seed, and **no shuffling by geography** for the
headline numbers — a random split is the honest one to report first, because a
spatial split would flatter the model by letting it memorise neighbours.

If you do add a spatial holdout (held-out regions), report it as a second
number rather than replacing the first, and say what it tests.

In [ ]:
# from sklearn.model_selection import train_test_split
#
# X, y, num_cols, cat_cols = mp2.split_features(df, TARGET, drop=LEAKY)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.25, stratify=y, random_state=42
# )
# print(X_train.shape, X_test.shape)
# print("train base rate:", round(y_train.mean(), 4))
# print("test  base rate:", round(y_test.mean(), 4))

## The pipeline

`ColumnTransformer` → median-impute and scale numerics, mode-impute and
one-hot the categoricals → estimator. Same preprocessor, models swapped, so
the comparison is fair.

In [ ]:
# from sklearn.pipeline import Pipeline
#
# preprocessor = mp2.make_preprocessor(num_cols, cat_cols)
#
# pipes = {
#     name: Pipeline([("prep", preprocessor), ("model", est)])
#     for name, est in mp2.build_models().items()
# }
# print(list(pipes))

## Cross-validated model comparison

Use `StratifiedKFold` and score *inside* the pipeline — never on data the
preprocessor has already seen.

In [ ]:
# from sklearn.model_selection import StratifiedKFold, cross_validate
#
# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
# scoring = {"recall": "recall", "pr_auc": "average_precision",
#            "roc_auc": "roc_auc", "accuracy": "accuracy"}
#
# rows = []
# for name, pipe in pipes.items():
#     res = cross_validate(pipe, X_train, y_train, cv=cv,
#                          scoring=scoring, n_jobs=-1)
#     rows.append({
#         "model": name,
#         "recall_pos": round(res["test_recall"].mean(), 4),
#         "pr_auc": round(res["test_pr_auc"].mean(), 4),
#         "roc_auc": round(res["test_roc_auc"].mean(), 4),
#         "accuracy": round(res["test_accuracy"].mean(), 4),
#     })
# cv_table = pd.DataFrame(rows).sort_values("recall_pos", ascending=False)
# print(cv_table.to_string(index=False))

## TODO — tune the chosen model

Pick the model on cross-validated recall/PR-AUC, then tune it. Search inside
the pipeline so the preprocessing is refit on each fold.

In [ ]:
# from sklearn.model_selection import GridSearchCV
#
# param_grid = {
#     "model__n_estimators": [200, 400, 800],
#     "model__min_samples_leaf": [1, 2, 5],
#     "model__max_features": ["sqrt", 0.3],
# }
#
# gs = GridSearchCV(pipes["forest"], param_grid, cv=cv,
#                   scoring="average_precision", n_jobs=-1)
# gs.fit(X_train, y_train)
# print(gs.best_params_)
# print(round(gs.best_score_, 4))

## Feature-set comparison — the second axis

Same model, three feature sets. This is where the project's question gets its
quantitative answer.

1. **Area profile only** — the socio-economic and housing variables
2. **+ cluster label** — the data-driven grouping
3. **+ administrative area** — regional council, then territorial authority

If the cluster label lifts recall and the administrative labels do not, that is
the finding. Report the deltas, not just the levels.

In [ ]:
# TODO: build the three feature sets and score the chosen model on each.
# Keep IDs and administrative names out of sets 1 and 2 — that is the point.
#
# feature_sets = {
#     "area profile": (NUMERIC, CATEGORICAL),
#     "+ cluster": (NUMERIC + ["cluster"], CATEGORICAL),
#     "+ region": (NUMERIC, CATEGORICAL + ["regional_council"]),
#     "+ TA": (NUMERIC, CATEGORICAL + ["territorial_authority"]),
# }

## Interpretation

What drives the prediction? Report it cautiously — this is association, not
causation, and the README must say so in those words.

In [ ]:
# TODO: permutation importance on the test set (model-agnostic, works for any model)
# from sklearn.inspection import permutation_importance
#
# imp = permutation_importance(fitted, X_test, y_test,
#                              scoring="average_precision",
#                              n_repeats=20, random_state=42, n_jobs=-1)
# order = np.argsort(imp.importances_mean)[::-1][:15]
# for i in order:
#     print(f"{X_test.columns[i]:40s} {imp.importances_mean[i]:+.4f}")

## Error analysis

Where does the model fail, and does it fail in a pattern? A model that misses
crowding in exactly the areas where crowding is worst is worse than its recall
score suggests — and that is worth saying.

In [ ]:
# TODO:
# - confusion matrix at your chosen threshold
# - false negatives by region and by cluster
# - threshold sensitivity: recall vs precision as the threshold moves
# - map the residuals spatially using the GeoPackage

## Class imbalance

Module 5 material. If crowding prevalence is low, run the same pipeline without
`class_weight="balanced"` and report both. Then you can say what the imbalance
handling actually bought, rather than assuming it helped.

In [ ]:
# TODO: same model, class_weight=None, compare recall and precision

## Save the results table

Notebook 05 reads this and writes the README numbers. Nothing gets typed from
memory into the README.

In [ ]:
# results.to_csv(mp2.OUT / "model_results.csv", index=False)
# print("written model_results.csv:", len(results), "rows")

## Limitations to carry forward

- **Ecological fallacy.** Area profile does not determine a household's
  outcome. State this in the README as the leading caveat.
- **Circularity.** Income and tenure sit close to the definition of crowding.
- **Suppression.** Median imputation on suppressed cells shrinks variance and
  biases toward large areas.
- **No causal claim.** Coefficients and importances are associations in one
  census snapshot.